In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [3]:
# 한글 폰트 설정 (Windows 기준)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [5]:
# 데이터 파일 경로
file_path = 'C:\\ai_x\\source\\JikFam\\data\\배추_이상치제거_주간기준_등급코드.csv'
output_dir = 'C:\\ai_x\\source\\JikFam\\eda_results'
os.makedirs(output_dir, exist_ok=True)

In [7]:
# 데이터 불러오기
try:
    df = pd.read_csv(file_path, encoding='cp949')
except FileNotFoundError:
    print(f"오류: 파일 '{file_path}'을(를) 찾을 수 없습니다.")
    # exit() 대신 노트북 환경에 맞게 중단 메시지 출력
    raise

## 1. 기초 통계 분석

In [8]:
print("--- 기초 통계량 ---")
print(df.describe())
print("--- 데이터 정보 ---")
df.info()

--- 기초 통계량 ---
           품목코드           품종코드           등급코드        총금액(원)       총거래량(kg)  \
count  786648.0  786648.000000  786648.000000  7.866480e+05  786648.000000   
mean        1.0      31.001344      12.195517  2.040985e+06    2717.420305   
std         0.0      41.360039       0.861605  8.588030e+06   10099.190784   
min         1.0       0.000000      11.000000  1.000000e+02       0.050000   
25%         1.0       4.000000      11.000000  1.008000e+05     100.000000   
50%         1.0       8.000000      12.000000  3.900000e+05     420.000000   
75%         1.0      99.000000      13.000000  1.400000e+06    1980.000000   
max         1.0      99.000000      20.000000  7.100456e+08  784070.000000   

             평균단가(원)      주간평균단가(원)         직팜산지코드  휴일여부  명절지수  작기정보  \
count  786648.000000  786648.000000  786648.000000   0.0   0.0   0.0   
mean     1252.033021    1251.136350    1066.727544   NaN   NaN   NaN   
std      1191.616128     614.944698      61.693954   NaN   NaN   N

In [9]:
# 데이터 결측치 확인
print("--- 결측치 확인 ---")
print(df.isnull().sum())

--- 결측치 확인 ---
주차                   0
연월일                  0
품목코드                 0
품목명                  0
품종코드                 0
품종명                  0
등급코드                 0
등급이름                 0
총금액(원)               0
총거래량(kg)             0
평균단가(원)              0
주간평균단가(원)            0
직팜산지코드               0
휴일여부            786648
명절지수            786648
작기정보            786648
일평균기온             1557
최고기온              1557
최저기온              1557
평균상대습도            1557
강수량(mm)           1557
1시간최고강수량(mm)      1557
dtype: int64


## 2. 데이터 전처리 (날짜 및 가격 컬럼 식별)

In [ ]:
# 날짜 데이터를 datetime 형식으로 변환
if '연도' in df.columns and '주' in df.columns:
    df['날짜'] = pd.to_datetime(df['연도'].astype(str) + df['주'].astype(str).str.zfill(2) + '1', format='%Y%W%w')
    date_col = '날짜'
elif 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])
    date_col = 'date'
else:
    date_col = df.columns[0]
    try:
        df[date_col] = pd.to_datetime(df[date_col])
    except (ValueError, TypeError):
        print(f"'{date_col}' 열을 날짜 형식으로 변환할 수 없습니다. 시계열 그래프를 생성하지 않습니다.")
        date_col = None

In [ ]:
# 가격 컬럼 확인
price_col = None
for col in ['가격', 'price', 'avg_price', '평균가격']:
    if col in df.columns:
        price_col = col
        break
if price_col is None:
    numeric_cols = df.select_dtypes(include='number').columns
    if len(numeric_cols) > 0:
        price_col = numeric_cols[-1]
        print(f"가격 컬럼을 찾지 못해 숫자형 마지막 컬럼인 '{price_col}'을 가격으로 간주합니다.")
    else:
        print("가격으로 추정할 숫자형 컬럼이 없습니다. 일부 시각화가 제한될 수 있습니다.")

## 3. 시각화

In [ ]:
# 시계열 그래프
if date_col and price_col:
    plt.figure(figsize=(15, 7))
    sns.lineplot(data=df, x=date_col, y=price_col)
    plt.title('시간에 따른 가격 변동')
    plt.xlabel('날짜')
    plt.ylabel('가격')
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, 'price_timeseries.png'))
    plt.show()
    plt.close()
    print("'price_timeseries.png' 저장 완료")

In [ ]:
# 가격 분포 (히스토그램 및 박스 플롯)
if price_col:
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    sns.histplot(df[price_col], kde=True)
    plt.title('가격 분포 (히스토그램)')

    plt.subplot(1, 2, 2)
    sns.boxplot(y=df[price_col])
    plt.title('가격 분포 (박스 플롯)')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'price_distribution.png'))
    plt.show()
    plt.close()
    print("'price_distribution.png' 저장 완료")

In [ ]:
# 페어 플롯 (숫자형 변수 간 관계)
numeric_df = df.select_dtypes(include=['number'])
if not numeric_df.empty:
    # 데이터가 너무 크면 샘플링하여 그리는 것이 좋음
    sample_df = numeric_df.head(1000) if len(numeric_df) > 1000 else numeric_df
    sns.pairplot(sample_df)
    plt.suptitle('숫자형 변수 간 산점도 행렬', y=1.02)
    plt.savefig(os.path.join(output_dir, 'pairplot.png'))
    plt.show()
    plt.close()
    print("'pairplot.png' 저장 완료")

## 4. 상관 관계 분석

In [ ]:
if not numeric_df.empty:
    plt.figure(figsize=(12, 10))
    corr_matrix = numeric_df.corr()
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=.5)
    plt.title('숫자형 변수 간 상관관계 히트맵')
    plt.savefig(os.path.join(output_dir, 'correlation_heatmap.png'))
    plt.show()
    plt.close()
    print("'correlation_heatmap.png' 저장 완료")